## LG Aimers 9기 — F 경기 분포 변화 대응 forward CV (Colab)

현재 **LightGBM pitchmix_shape OOF와 65:35 logit 혼합비를 고정**하고 CatBoost의 F 학습 정책만 바꾼다. `game_type` 열 포함 여부, F 최근 가중치, 최근 F 학습 창, F-only 전문가를 분리 비교한다. 모든 2022·2023·2024 holdout은 해당 시즌보다 과거 데이터만 학습하며, 이 노트북은 제출 ZIP을 만들지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

candidates = [
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/LG Aimers 9th"),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "experiments/07_f_shift_cv.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "LG_aimers_9th 폴더 또는 experiments/07_f_shift_cv.py를 찾지 못했습니다."
    )

required = [
    PROJECT_ROOT / "data/train.csv",
    PROJECT_ROOT / "src/features.py",
    PROJECT_ROOT / "src/evaluation.py",
    PROJECT_ROOT / "src/metrics.py",
    PROJECT_ROOT / "experiments/07_f_shift_cv.py",
    PROJECT_ROOT / "outputs/feature_ablation_v1/latest_run.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("train.csv size (GB):", round((PROJECT_ROOT / "data/train.csv").stat().st_size / 1e9, 3))

### 1. CatBoost와 A100 확인

이 실험은 새 CatBoost를 24회 학습하므로 A100 런타임을 권장한다. Colab 기본 NumPy·pandas는 다시 설치하지 않고 CatBoost만 맞춘다. 설치 뒤 import 문제가 생기면 런타임을 다시 시작하고 0번 셀부터 실행한다.

In [ ]:
install = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "-q", "catboost==1.2.8",
    ],
    capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout[-3000:])
    print(install.stderr[-3000:])
    raise RuntimeError("CatBoost 설치에 실패했습니다. 위 로그를 확인하세요.")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import catboost

gpu_binary = shutil.which("nvidia-smi")
if gpu_binary is None:
    raise RuntimeError("GPU가 없습니다. Colab 런타임 유형에서 A100 GPU를 선택하세요.")
gpu = subprocess.run(
    [gpu_binary, "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
)
print("GPU:", gpu.stdout.strip())
print("CatBoost:", catboost.__version__)
print("pandas:", pd.__version__, "| NumPy:", np.__version__)

### 2. 05 OOF와 F 분포 사전 점검

기준 LightGBM은 05의 `pitchmix_shape`, 기준 CatBoost는 05의 `all_new` OOF다. 두 파일의 행·타깃·시즌·경기 유형은 실행 스크립트에서도 다시 완전 일치 검사한다. 아래 성공률은 모델 선택 결과가 아니라 F 분포 변화가 실제로 큰지 확인하는 입력 진단이다.

In [ ]:
FEATURE_OUTPUT_ROOT = PROJECT_ROOT / "outputs/feature_ablation_v1"
with (FEATURE_OUTPUT_ROOT / "latest_run.json").open(encoding="utf-8") as handle:
    feature_latest = json.load(handle)
recorded_source = Path(feature_latest["run_dir"])
SOURCE_RUN_DIR = next(
    (p for p in [recorded_source, FEATURE_OUTPUT_ROOT / recorded_source.name] if p.exists()),
    None,
)
if SOURCE_RUN_DIR is None:
    raise FileNotFoundError("05 최신 run 폴더를 찾지 못했습니다.")

source_oof = [
    SOURCE_RUN_DIR / "oof_pitchmix_shape.csv.gz",
    SOURCE_RUN_DIR / "oof_all_new.csv.gz",
]
missing_oof = [str(path) for path in source_oof if not path.exists()]
if missing_oof:
    raise FileNotFoundError(f"05 OOF 파일이 없습니다: {missing_oof}")

rate_history = pd.read_csv(
    PROJECT_ROOT / "data/train.csv",
    usecols=["season", "game_type", "control_success"],
)
rate_table = (
    rate_history.groupby(["season", "game_type"], observed=True)["control_success"]
    .agg(n="size", target_rate="mean")
    .reset_index()
)
display(rate_table.pivot(index="season", columns="game_type", values="target_rate"))
print("SOURCE_RUN_DIR:", SOURCE_RUN_DIR)
print({path.name: round(path.stat().st_size / 1e6, 1) for path in source_oof})
del rate_history

### 3. 실험군과 선택 기준 고정

`current_all_nogt`는 다시 학습하지 않고 기존 OOF를 그대로 사용한다. `retrain_all_nogt`와 `all_with_gt`를 같은 실행 환경에서 함께 학습해 GPU 재학습 변동과 `game_type` 열 효과를 구분한다. `shared_*`는 R/F를 한 모델에서 학습하되 F 표본 정책을 바꾸고, `expert_*`는 R 예측을 기준과 완전히 동일하게 유지한 채 F 행만 F-only 모델로 교체한다.

In [ ]:
VALID_SEASONS = [2022, 2023, 2024]
THREADS = 8
CATBOOST_ITERATIONS = 85
FORCE_RERUN = False

policy_plan = pd.DataFrame([
    ["current_all_nogt", "기준", "과거 R/F 전체, half-life 0.5, game_type 제외"],
    ["retrain_all_nogt", "재학습 대조", "기준 레시피를 같은 환경에서 재학습; 자동 선택 제외"],
    ["all_with_gt", "공유", "기준과 동일하되 game_type 포함"],
    ["r_only", "공유", "R만 학습하여 R/F 모두 예측"],
    ["shared_f_fast_decay", "공유", "R half-life 0.5, F half-life 0.25"],
    ["shared_latest_f1", "공유", "과거 R 전체 + 직전 F 1시즌"],
    ["expert_f_fast_decay", "F 전문가", "R 기준 유지 + 과거 F half-life 0.25"],
    ["expert_f_latest1", "F 전문가", "R 기준 유지 + 직전 F 1시즌"],
    ["expert_f_latest2", "F 전문가", "R 기준 유지 + 직전 F 2시즌"],
] , columns=["policy", "family", "training_rule"])
display(policy_plan)

자동 채택은 다음 조건을 모두 만족해야 한다.

- 2023+2024 pooled 전체 Brier와 F Brier가 기준보다 개선
- 2023과 2024 각각에서 전체·R·F Brier가 어느 하나도 기준보다 악화되지 않음
- 조건을 만족하는 후보가 없으면 현재 정책 유지

`retrain_all_nogt`는 GPU 재학습 변동을 재는 대조군이라 점수가 좋아도 자동 선택하지 않는다. 2022는 F 급변 이전 구간이므로 진단표에는 남기지만 채택 조건에는 넣지 않는다. 이는 2023·2024를 보고 보정 파라미터를 맞추는 실험이 아니며, 모든 예측 모델은 각 holdout보다 이전 시즌만 사용한다.

### 4. F 정책 forward CV 실행

첫 실행은 A100에서도 여러 CatBoost fold를 순차 학습한다. 각 policy/fold 결과가 Drive에 캐시되므로 중단 후 같은 셀을 다시 실행하면 완료된 fold는 재사용한다. 처음부터 다시 학습할 때만 `FORCE_RERUN=True`로 바꾼다.

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs/f_shift_cv_v1"
command = [
    sys.executable,
    str(PROJECT_ROOT / "experiments/07_f_shift_cv.py"),
    "--train-path", str(PROJECT_ROOT / "data/train.csv"),
    "--feature-output-root", str(FEATURE_OUTPUT_ROOT),
    "--source-run-dir", str(SOURCE_RUN_DIR),
    "--output-dir", str(OUTPUT_ROOT),
    "--valid-seasons", *map(str, VALID_SEASONS),
    "--threads", str(THREADS),
    "--catboost-iterations", str(CATBOOST_ITERATIONS),
    "--catboost-task-type", "GPU",
]
if FORCE_RERUN:
    command.append("--force-rerun")

print("Running:", " ".join(command))
subprocess.run(command, check=True)

### 5. 실행 결과와 기준 조합 무결성 확인

In [ ]:
with (OUTPUT_ROOT / "latest_run.json").open(encoding="utf-8") as handle:
    latest = json.load(handle)
recorded_run = Path(latest["run_dir"])
RUN_DIR = next(
    (p for p in [recorded_run, OUTPUT_ROOT / recorded_run.name] if p.exists()),
    None,
)
if RUN_DIR is None:
    raise FileNotFoundError("07 결과 run 폴더를 찾지 못했습니다.")
summary = latest["summary"]

integrity = pd.DataFrame([
    ["source rows", summary["input_audit"]["rows"]],
    ["fixed LightGBM", summary["fixed_lgbm_component"]],
    ["fixed blend", summary["fixed_blend"]],
    ["CatBoost task", summary["catboost_task_type"]],
    ["base recent Brier", summary["base_recent_brier"]],
    ["base recent BSS raw", summary["base_recent_bss_raw"]],
] , columns=["check", "value"])
display(integrity)
print("RUN_DIR:", RUN_DIR)

### 6. 정책별 핵심 결과

Brier와 `delta_*`는 낮을수록 좋고 BSS는 높을수록 좋다. `delta_recent_brier_x1e6`는 작은 차이를 보기 쉽게 백만 배 한 값이며, 음수면 기준보다 개선이다. 엄격 조건을 통과하지 않은 상위 후보도 원인 분석에는 쓰지만 곧바로 최종 학습에 반영하지 않는다.

In [ ]:
policy_summary = pd.read_csv(RUN_DIR / "policy_summary.csv")
policy_summary["delta_recent_brier_x1e6"] = (
    1e6 * policy_summary["delta_recent_brier_vs_base"]
)
policy_summary["delta_recent_f_brier_x1e6"] = (
    1e6 * policy_summary["delta_recent_f_brier_vs_base"]
)
core_columns = [
    "policy", "recent_brier", "recent_bss_raw", "recent_mean_gap",
    "delta_recent_brier_x1e6", "delta_recent_f_brier_x1e6",
    "worst_recent_fold_delta", "selection_eligible", "passes_strict_selection",
]
display(policy_summary.sort_values("recent_brier")[core_columns].reset_index(drop=True))
print("Selected:", summary["selected_policy"])
print("Strict safe improving:", summary["strict_safe_improving_policies"])

### 7. 폴드별 전체·R·F 안정성

2022 성능은 급변 전 진단이고, 채택 판단의 핵심은 2023·2024 방향 일치다. 전체 개선이 F에서만 발생하면서 R을 훼손했는지, 혹은 2023만 좋아지고 2024가 나빠졌는지 함께 확인한다.

In [ ]:
policy_metrics = pd.read_csv(RUN_DIR / "policy_metrics.csv")
blend_folds = policy_metrics[
    policy_metrics["model"].eq("fixed_logit_65_35")
    & policy_metrics["partition"].eq("fold")
]
fold_table = blend_folds.pivot_table(
    index="policy", columns=["valid_season", "scope"], values="brier"
)
display(fold_table.loc[policy_summary.sort_values("recent_brier")["policy"]])

In [ ]:
base_name = "current_all_nogt"
recent = policy_summary.sort_values("recent_brier").copy()
plot_columns = [
    "delta_brier_r_2023_vs_base", "delta_brier_f_2023_vs_base",
    "delta_brier_r_2024_vs_base", "delta_brier_f_2024_vs_base",
]
plot_frame = recent.set_index("policy")[plot_columns] * 1e6
ax = plot_frame.plot(kind="bar", figsize=(14, 5))
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("Brier delta vs current × 1e6 (lower is better)")
ax.set_title("R/F fold stability relative to current policy")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### 8. `game_type` 열 자체의 효과

`retrain_all_nogt - current_all_nogt`는 동일 레시피 재학습 변동을 보여준다. `all_with_gt - retrain_all_nogt`가 같은 실행 환경·학습 행·감쇠 규칙에서 본 `game_type` 열의 직접 효과다. `r_only`와 최근 F 정책은 학습 데이터 자체가 달라 별도로 해석한다.

In [ ]:
gt_compare = policy_summary[
    policy_summary["policy"].isin(["current_all_nogt", "retrain_all_nogt", "all_with_gt"])
].copy()
gt_columns = [
    "policy", "recent_brier", "recent_r_brier", "recent_f_brier",
    "brier_r_2023", "brier_f_2023", "brier_r_2024", "brier_f_2024",
    "delta_recent_brier_vs_base", "passes_strict_selection",
]
display(gt_compare[gt_columns])

gt_metric_source = policy_metrics[
    policy_metrics["model"].eq("fixed_logit_65_35")
    & policy_metrics["policy"].isin(["retrain_all_nogt", "all_with_gt"])
    & policy_metrics["partition"].isin(["recent_pooled", "fold"])
].copy()
gt_metric_source["evaluation_season"] = (
    gt_metric_source["valid_season"].astype("Int64").astype("string").fillna("2023+2024")
)
gt_metric_rows = gt_metric_source.pivot_table(
    index=["partition", "evaluation_season", "scope"],
    columns="policy", values="brier",
).reset_index()
gt_metric_rows["with_gt_minus_matched_nogt_x1e6"] = 1e6 * (
    gt_metric_rows["all_with_gt"] - gt_metric_rows["retrain_all_nogt"]
)
display(gt_metric_rows.sort_values(["partition", "evaluation_season", "scope"]))

### 9. F-only 전문가와 paired loss

F-only 정책은 R에서 기준 예측을 그대로 복사하므로 R Brier가 정확히 같아야 한다. Paired loss 평균은 같은 행에서 후보 제곱오차 minus 기준 제곱오차이며 음수가 좋다. `naive_standard_error`는 경기 내 상관을 무시하므로 유의확률로 해석하지 않는다.

In [ ]:
expert_names = policy_summary.loc[
    policy_summary["policy"].str.startswith("expert_"), "policy"
].tolist()
expert_r_delta_columns = [
    "policy", "delta_brier_r_2022_vs_base",
    "delta_brier_r_2023_vs_base", "delta_brier_r_2024_vs_base",
]
display(policy_summary[policy_summary["policy"].isin(expert_names)][expert_r_delta_columns])

paired = pd.read_csv(RUN_DIR / "paired_loss_deltas.csv")
paired_focus = paired[
    paired["partition"].eq("recent_pooled")
    & paired["scope"].isin(["all", "F"])
].sort_values(["scope", "mean_loss_delta_vs_base"])
display(paired_focus.reset_index(drop=True))

### 10. 예측 변화량과 학습 상태

상관도가 높아도 평균 절대 변화가 특정 F 구간에 집중되면 Brier가 달라질 수 있다. 학습 메타데이터에서는 정책별 학습 행 수, F 비중, 유효 표본 크기와 실행 시간을 확인한다.

In [ ]:
correlations = pd.read_csv(RUN_DIR / "prediction_correlations.csv")
correlation_focus = correlations[
    correlations["model"].eq("blend")
    & correlations["partition"].eq("recent_pooled")
    & correlations["scope"].isin(["all", "F"])
].sort_values(["scope", "mean_abs_prediction_delta_vs_base"], ascending=[True, False])
display(correlation_focus.reset_index(drop=True))

training = pd.read_csv(RUN_DIR / "training_metadata.csv")
training_columns = [
    "policy", "valid_season", "prediction_scope", "train_rows",
    "train_f_fraction", "train_target_rate", "effective_sample_size",
    "feature_count", "train_seconds", "task_type",
]
display(training[training_columns].sort_values(["policy", "valid_season"]).reset_index(drop=True))

### 11. 자동 판정과 다음 단계

07의 선택 결과는 리더보드용 제출물이 아니라 최종 학습 레시피 후보다. 비기준 정책이 엄격 조건을 통과하면 실행 결과를 검토한 뒤 07b 최종 학습에 재현한다. 기준이 유지되면 F 처리만으로는 안정적 개선 근거가 없다는 뜻이므로 제출 2를 유지하고 ExtraTrees·신경망 등 다른 모델 계열로 다양성을 넓힌다.

In [ ]:
decision = {
    "selected_policy": summary["selected_policy"],
    "strict_safe_improving_policies": summary["strict_safe_improving_policies"],
    "delta_recent_brier_vs_base": summary["selected_delta_recent_brier_vs_base"],
    "selected_recent_bss_raw": summary["selected_recent_bss_raw"],
    "next_step": summary["next_step"],
}
display(pd.DataFrame(decision.items(), columns=["item", "value"]))
if summary["selected_policy"] == summary["base_policy"]:
    print("판정: 현재 F 정책 유지. 07 결과만으로 새 제출을 만들지 않습니다.")
else:
    print("판정: 엄격 조건 통과 후보 발견. 결과 검토 후 07b 최종 학습으로 이동합니다.")

### 12. 검토용 결과 ZIP 생성

In [ ]:
review_names = [
    "summary.json",
    "configuration.json",
    "policy_manifest.csv",
    "policy_summary.csv",
    "policy_metrics.csv",
    "paired_loss_deltas.csv",
    "prediction_correlations.csv",
    "training_metadata.csv",
    "feature_importance.csv",
    "f_rate_history.csv",
]
REVIEW_ZIP = OUTPUT_ROOT / "f_shift_review.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        path = RUN_DIR / name
        if not path.exists():
            raise FileNotFoundError(f"검토 파일이 없습니다: {path}")
        archive.write(path, arcname=name)

print("검토 ZIP:", REVIEW_ZIP)
print("크기 (MB):", round(REVIEW_ZIP.stat().st_size / 1e6, 2))
print("전달 파일: 실행 결과가 저장된 07_f_shift_cv.ipynb + f_shift_review.zip")